So personally, I think the steps that need be to taken are:
1. Load pdf
2. Extract text
3. Chunking
4. Create embeddings
5. Retrieval
6. Prompting
7. Testing

### Loading pdf and text extraction
I m gonna be using the pdf of book "Who moved my Cheese"

In [1]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pypdf

In [3]:
reader = pypdf.PdfReader("../data/documents/who_moved_my_cheese.pdf")
page = reader.pages[0]
print(page.extract_text()) ## ->> extracts texts of all orientations

 
Who 
Moved 
My 
Cheese? 
 
 
 
An Amazing Way to Deal with Change in Your 
Work and in Your Life 
 
 
Dr Spencer Johnson 
 
 
 
 
Foreword by KENNETH BLANCHARD Ph.D. 
 
 


In [4]:
print(page.extract_text(0)) ## extracts texts of 0 degrees

 
Who 
Moved 
My 
Cheese? 
 
 
 
An Amazing Way to Deal with Change in Your 
Work and in Your Life 
 
 
Dr Spencer Johnson 
 
 
 
 
Foreword by KENNETH BLANCHARD Ph.D. 
 
 


Now i think the first step is to extract all the text from the pdf for chunking

In [5]:
pages = []
for i,page in enumerate(reader.pages):
    pages.append({
        "page_no" : i+1,
        "text" : page.extract_text()
    })

### Cleaning unnecessary things


In [6]:
for page in pages:
    text = page["text"]

    text = text.replace("\n", " ")
    text = " ".join(text.split())

    page["text"] = text

In [7]:
print(pages[:2])

[{'page_no': 1, 'text': 'Who Moved My Cheese? An Amazing Way to Deal with Change in Your Work and in Your Life Dr Spencer Johnson Foreword by KENNETH BLANCHARD Ph.D.'}, {'page_no': 2, 'text': "The Story Behind The Story by Kenneth Blanchard, Ph.D. I am thrilled to be telling you “the story behind the story” of Who Moved My Cheese? because it means the book has now been written, and is available for all of us to read, enjoy and share with others. This is something I've wanted to see happen ever since I first heard Spencer Johnson tell his great “Chees e” story, years ago, be fore we wrote our book The One Minute Manager together. I remember thinking then how good the story was and how helpful it would be to me from that moment on. Who Moved My Cheese? is a story about change that takes place in a Maze where four amusing characters look for “Cheese”-cheese being a metaphor for what we want to have in life, whether it is a job, a relationship, money, a big house, freedom, health, recognit

### Chunking
(heart of a RAG) ->> we will be using the simplest chunking strategy : fixed size chunks with overlap

overlap so we can retain some info of previous as well
```text
Chunk Size = 500
Overlap   = 100

Chunk 1: [0 ------------------------------------------- 500]
Chunk 2:             [400 ------------------------------------------- 900]
Chunk 3:                             [800 ------------------------------------------- 1300]
```

In [8]:
chunked_text = []
chunk_size = 500
overlap = 100

for page in pages:
    text = page["text"]
    page_num = page["page_no"]

    start = 0
    chunk_id = 1

    while start<len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        chunked_text.append({
            "chunk_id" : chunk_id,
            "page_no" : page_num,
            "chunked_text" : chunk_text
        })

        start = start + (chunk_size - overlap)
        chunk_id = chunk_id + 1


Rn im using character based chunking, which ik is bad. Better alternatives :
1. Token based
2. Recursive
3. Semantic

In [9]:
chunked_text[0]

{'chunk_id': 1,
 'page_no': 1,
 'chunked_text': 'Who Moved My Cheese? An Amazing Way to Deal with Change in Your Work and in Your Life Dr Spencer Johnson Foreword by KENNETH BLANCHARD Ph.D.'}

### Creating Embeddings
here we are generating sentence embeddings, so model can search when needed /n
also, this will be done by a model ''' Sentence - Transformer '''

In [10]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

d:\Anaconda3\envs\torchgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 828.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
embedding = model.encode("Hello World")

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [13]:
embedding

array([-3.44772898e-02,  3.10232006e-02,  6.73499424e-03,  2.61089318e-02,
       -3.93619724e-02, -1.60302490e-01,  6.69239834e-02, -6.44147908e-03,
       -4.74505424e-02,  1.47588309e-02,  7.08753243e-02,  5.55274785e-02,
        1.91933457e-02, -2.62513310e-02, -1.01094618e-02, -2.69405413e-02,
        2.23074127e-02, -2.22266410e-02, -1.49692625e-01, -1.74931269e-02,
        7.67623028e-03,  5.43523021e-02,  3.25448788e-03,  3.17259803e-02,
       -8.46214145e-02, -2.94059366e-02,  5.15956618e-02,  4.81241085e-02,
       -3.31480382e-03, -5.82792200e-02,  4.19692807e-02,  2.22107638e-02,
        1.28188819e-01, -2.23389510e-02, -1.16562163e-02,  6.29284382e-02,
       -3.28762941e-02, -9.12260115e-02, -3.11753694e-02,  5.26995398e-02,
        4.70348187e-02, -8.42030272e-02, -3.00561562e-02, -2.07448322e-02,
        9.51773021e-03, -3.72175244e-03,  7.34330248e-03,  3.93243618e-02,
        9.32740048e-02, -3.78855458e-03, -5.27420975e-02, -5.80582246e-02,
       -6.86441967e-03,  

In [15]:
for chunk in chunked_text:
    text = chunk["chunked_text"]
    embedding = model.encode(text)

    chunk["embedding"] = embedding

In [18]:
print(chunked_text[0].keys())

dict_keys(['chunk_id', 'page_no', 'chunked_text', 'embedding'])
